In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import average_precision_score, roc_auc_score

# 1. Load data and splits
curated = pd.read_csv("../data/curated_chembl.csv")
splits = json.loads(Path("../splits/split_indices.json").read_text())

# 2. Generate ECFP4 using the new API
# Locked: ECFP4 (radius=2), 2048 bits
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def generate_ecfp4(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = morgan_gen.GetFingerprint(mol)
    return list(fp)

curated["ecfp4"] = curated["canonical_smiles"].apply(generate_ecfp4)

# Check for any failures
missing_fps = curated["ecfp4"].isna().sum()
if missing_fps > 0:
    print(f"WARNING: {missing_fps} compounds failed to generate fingerprints.")
else:
    print("ECFP4 feature generation complete. No failures.")

# 3. Build matrices
def get_xy(indices):
    subset = curated.iloc[indices]
    X = np.array(subset["ecfp4"].tolist())
    y = (subset["activity_class"] == "active").astype(int).values
    return X, y

X_train, y_train = get_xy(splits["indices"]["train"])
X_val, y_val = get_xy(splits["indices"]["validation"])
X_test, y_test = get_xy(splits["indices"]["test"])

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

ECFP4 feature generation complete. No failures.
Train shape: (201, 2048)
Validation shape: (42, 2048)
Test shape: (44, 2048)


In [3]:
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import average_precision_score

# 1. Load CV folds (these indices are relative to the FULL curated dataset)
cv_folds_global = json.loads(Path("../splits/cv_scaffold_groups.json").read_text())["folds"]

# 2. Create a mapping from global indices to local train indices (0 to 200)
train_global_indices = splits["indices"]["train"]
global_to_local = {g_idx: l_idx for l_idx, g_idx in enumerate(train_global_indices)}

# 3. Map the folds to local indices
cv_folds_local = {}
for fold_name, global_idxs in cv_folds_global.items():
    # Only keep indices that are actually in the training set, and map them
    cv_folds_local[fold_name] = [global_to_local[g] for g in global_idxs if g in global_to_local]

# 4. Evaluate CV function
def evaluate_cv(model_func, X, y, folds):
    pr_aucs = []
    for fold_local_indices in folds.values():
        train_local_indices = [i for i in range(len(X)) if i not in fold_local_indices]
        
        model = model_func()
        model.fit(X[train_local_indices], y[train_local_indices])
        
        preds = model.predict_proba(X[fold_local_indices])[:, 1]
        pr_aucs.append(average_precision_score(y[fold_local_indices], preds))
    return np.mean(pr_aucs), np.std(pr_aucs)

# 5. Train and evaluate XGB
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Scale Pos Weight: {scale_pos_weight:.2f}")

xgb_func = lambda: xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, 
    scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss"
)
xgb_mean, xgb_std = evaluate_cv(xgb_func, X_train, y_train, cv_folds_local)

# 6. Train and evaluate RF
rf_func = lambda: RandomForestClassifier(
    n_estimators=100, max_depth=5, class_weight="balanced", random_state=42
)
rf_mean, rf_std = evaluate_cv(rf_func, X_train, y_train, cv_folds_local)

print(f"XGB CV PR-AUC: {xgb_mean:.3f} ± {xgb_std:.3f}")
print(f"RF CV PR-AUC:  {rf_mean:.3f} ± {rf_std:.3f}")

# 7. Select the best model
if xgb_mean > rf_mean:
    print("Selected Model: XGBoost")
    selected_model = xgb_func()
else:
    print("Selected Model: Random Forest")
    selected_model = rf_func()

# Fit the selected model on the full training set
selected_model.fit(X_train, y_train)

Scale Pos Weight: 8.14


c:\Users\aymen\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\aymen\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\aymen\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\aymen\anaconda3\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


XGB CV PR-AUC: 0.261 ± 0.221
RF CV PR-AUC:  0.264 ± 0.244
Selected Model: Random Forest


,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [4]:
def evaluate_split(model, X, y, name):
    preds = model.predict_proba(X)[:, 1]
    pr_auc = average_precision_score(y, preds)
    roc_auc = roc_auc_score(y, preds)
    
    # Top-N recovery (N = number of actives)
    n_actives = sum(y)
    top_n_preds = np.argsort(preds)[-n_actives:]
    top_n_hits = sum(y[top_n_preds])
    top_n_recovery = top_n_hits / n_actives
    
    print(f"[{name}] N_actives: {n_actives}, N_inactives: {len(y)-n_actives}")
    print(f"  PR-AUC: {pr_auc:.3f}, ROC-AUC: {roc_auc:.3f}")
    print(f"  Top-{n_actives} Recovery: {top_n_recovery:.3f} ({top_n_hits}/{n_actives})")
    return pr_auc, roc_auc, top_n_recovery

print("--- Validation ---")
evaluate_split(selected_model, X_val, y_val, "Validation")
print("\n--- Test (Headline) ---")
evaluate_split(selected_model, X_test, y_test, "Test")

--- Validation ---
[Validation] N_actives: 4, N_inactives: 38
  PR-AUC: 0.887, ROC-AUC: 0.987
  Top-4 Recovery: 0.750 (3/4)

--- Test (Headline) ---
[Test] N_actives: 5, N_inactives: 39
  PR-AUC: 0.370, ROC-AUC: 0.533
  Top-5 Recovery: 0.400 (2/5)


(0.36996658312447783, 0.5333333333333333, np.float64(0.4))

In [5]:
import joblib
import json
import pandas as pd
from pathlib import Path

# 1. Save the trained Random Forest model
Path("../models").mkdir(exist_ok=True)
joblib.dump(selected_model, "../models/selected_model_rf.pkl")
print("Model saved to ../models/selected_model_rf.pkl")

# 2. Save the model config (decision-log data)
model_config = {
    "schema_version": "1.0.0",
    "selected_model": "RandomForestClassifier",
    "xgb_cv_pr_auc": float(xgb_mean),
    "rf_cv_pr_auc": float(rf_mean),
    "params": {
        "n_estimators": 100,
        "max_depth": 5,
        "class_weight": "balanced",
        "random_state": 42
    }
}
Path("../models/model_config.json").write_text(json.dumps(model_config, indent=2))
print("Model config saved to ../models/model_config.json")

# 3. Write the locked evaluation table
# Note: applying small-N flags per protocol.md §5.3
eval_data = [
    {
        "model": "Dummy",
        "split": "test",
        "pr_auc": 0.113, # Baseline positive class fraction (5/44)
        "roc_auc": 0.500,
        "top_n_recovery": 0.200, # Random expectation
        "notes": "Baseline"
    },
    {
        "model": "Similarity-only",
        "split": "test",
        "pr_auc": None, # To be filled in Day 3 (needs P2's similarity script)
        "roc_auc": None,
        "top_n_recovery": None,
        "notes": "Pending P2"
    },
    {
        "model": "Selected RF",
        "split": "validation",
        "pr_auc": 0.887,
        "roc_auc": 0.987,
        "top_n_recovery": 0.750,
        "notes": "N_actives=4, ef_unstable=True, ci_not_reported=True"
    },
    {
        "model": "Selected RF",
        "split": "test",
        "pr_auc": 0.370,
        "roc_auc": 0.533,
        "top_n_recovery": 0.400,
        "notes": "N_actives=5, top_recovery_mode='top_N', ef_unstable=True, ci_not_reported=True"
    }
]

df_eval = pd.DataFrame(eval_data)
df_eval.to_csv("../results/evaluation_table.csv", index=False)
print("Evaluation table saved to ../results/evaluation_table.csv")

Model saved to ../models/selected_model_rf.pkl
Model config saved to ../models/model_config.json
Evaluation table saved to ../results/evaluation_table.csv


In [6]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
import numpy as np
import pandas as pd

# 1. Re-initialize the generator (same as Cell 1)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# 2. Get RDKit fingerprint objects for the training set
train_smiles = curated.iloc[splits["indices"]["train"]]["canonical_smiles"]
train_fps = [morgan_gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_smiles]

# 3. Calculate LOO training distances
train_distances = []
for i, fp in enumerate(train_fps):
    # Calculate Tanimoto similarity to all other training compounds
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.pop(i) # Remove self-similarity (which is 1.0)
    sims.sort(reverse=True)
    
    # Mean Jaccard distance (1 - Tanimoto) to the 5 nearest neighbors
    dist = np.mean([1 - s for s in sims[:5]])
    train_distances.append(dist)

# Locked threshold: 95th percentile of the training LOO distances
ad_threshold = np.percentile(train_distances, 95)
print(f"AD Threshold (95th percentile): {ad_threshold:.4f}\n")

# 4. Calculate AD for the 28 Rosemary compounds
rosemary = pd.read_csv("../data/cultivar_compound_catalogue.csv").drop_duplicates("compound_id")
ad_results = []

for _, row in rosemary.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is None:
        continue
    fp = morgan_gen.GetFingerprint(mol)
    
    # Calculate similarity to training set
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.sort(reverse=True)
    
    # Mean Jaccard distance to 5 nearest training neighbors
    dist = np.mean([1 - s for s in sims[:5]])
    
    flag = "within_AD" if dist <= ad_threshold else "extrapolation"
    ad_results.append({
        "compound_id": row["compound_id"], 
        "compound_name": row["compound_name"],
        "ad_distance": dist, 
        "ad_flag": flag
    })

df_ad = pd.DataFrame(ad_results)

# 5. Report results
print("AD Summary for Rosemary Catalogue:")
print(df_ad["ad_flag"].value_counts().to_string())
print("\nCompounds flagged as Extrapolation (Distance > Threshold):")
print(df_ad[df_ad["ad_flag"] == "extrapolation"][["compound_name", "ad_distance"]].sort_values("ad_distance", ascending=False).head(10).to_string(index=False))

# Save the results for Day 3 scoring
df_ad.to_csv("../results/ad_results.csv", index=False)
print("\nSaved AD results to ../results/ad_results.csv")

AD Threshold (95th percentile): 0.4638

AD Summary for Rosemary Catalogue:
ad_flag
extrapolation    28

Compounds flagged as Extrapolation (Distance > Threshold):
      compound_name  ad_distance
           Camphene     0.895230
    gamma-Terpinene     0.889970
            Myrcene     0.889007
Caryophyllene oxide     0.888031
        beta-Pinene     0.887331
            Camphor     0.886082
        1,8-Cineole     0.885842
     gamma-Eudesmol     0.885759
           Linalool     0.884064
       alpha-Pinene     0.881906

Saved AD results to ../results/ad_results.csv
